# Лабораторная работа 4. Часть 2 — Т-тесты

**ФИО:** Иноземцев Сергей Викторович  
**Группа:** ЕТ-216

Цель: проверить различия средних психологических показателей между юношами и девушками. Исходный файл называется `Психологи.xlsx` (в тексте задания — «Психология.xlsx»).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 25)
pd.set_option('display.max_rows', 25)
sns.set_theme(style='whitegrid', font='DejaVu Sans')
plt.rcParams.update({'figure.figsize': (9, 5), 'figure.dpi': 110})
DATA = Path('.')
FIG = Path('figures')
FIG.mkdir(exist_ok=True)

def show(name):
    plt.tight_layout()
    plt.savefig(FIG / (name + '.png'), bbox_inches='tight')
    plt.show()
    plt.close('all')


## 2. Гипотезы и выбор критерия

H₀: средние значения показателя у юношей и девушек одинаковы. H₁: средние различаются. Уровень значимости α=0,05. Группы независимы; используем двусторонний t-тест Уэлча (`equal_var=False`). Парный тест не подходит, поскольку строки не являются повторными измерениями одних людей.

Номер учащегося и код профиля исключены из психологических шкал. Для каждого из 15 показателей приведены t, исходное p-value и вывод при α=0,05; отдельно добавлена поправка Holm на множественные проверки, 95% ДИ разности и Cohen d. Неотклонение H₀ не доказывает равенство групп.

In [ ]:
def holm(pvalues):
    p=np.asarray(pvalues);order=np.argsort(p);adjusted=np.empty(len(p))
    adjusted[order]=np.minimum(1,np.maximum.accumulate((len(p)-np.arange(len(p)))*p[order]));return adjusted
def compare_groups(frame,group,cols,a,b):
    rows=[]
    for col in cols:
        x=frame.loc[frame[group].eq(a),col].dropna().to_numpy();y=frame.loc[frame[group].eq(b),col].dropna().to_numpy()
        t,p=stats.ttest_ind(x,y,equal_var=False)
        diff=x.mean()-y.mean();vx=x.var(ddof=1)/len(x);vy=y.var(ddof=1)/len(y);se=np.sqrt(vx+vy)
        dof=(vx+vy)**2/(vx*vx/(len(x)-1)+vy*vy/(len(y)-1));margin=stats.t.ppf(.975,dof)*se
        pooled=np.sqrt(((len(x)-1)*x.var(ddof=1)+(len(y)-1)*y.var(ddof=1))/(len(x)+len(y)-2))
        rows.append([col,len(x),len(y),x.mean(),y.mean(),diff,t,p,diff-margin,diff+margin,diff/pooled])
    r=pd.DataFrame(rows,columns=['показатель','n_A','n_B','mean_A','mean_B','A-B','t','p','CI_low','CI_high','Cohen_d'])
    r['значимо без поправки']=r.p<.05;r['p_Holm']=holm(r.p);r['вывод_0.05']=np.where(r.p_Holm<.05,'H0 отклонена','H0 не отклонена');return r


## Психологические показатели

In [ ]:
psy=pd.read_excel(DATA/'Психологи.xlsx');psy.columns=psy.columns.str.strip();psy['пол']=psy['пол'].str.strip().str.lower()
display(psy.head());display(psy['пол'].value_counts());display(psy.isna().sum())
psy_cols=[c for c in psy.select_dtypes('number').columns if c not in ['учащиеся','профиль ВУЗа']]
psy_tests=compare_groups(psy,'пол',psy_cols,'м','ж');display(psy_tests.sort_values('p'))
normal_psy=[]
for c in psy_cols:
    for g in ['м','ж']:
        x=psy.loc[psy['пол'].eq(g),c].dropna();normal_psy.append([c,g,len(x),stats.shapiro(x).pvalue])
display(pd.DataFrame(normal_psy,columns=['шкала','пол','n','p_Shapiro']))
psy_tests.to_csv('psychology_welch.csv',index=False)

Минимальное p у шкалы «экстраверсия- интроверсия», максимальное — «средний IQ». Это сила статистического свидетельства, не размер различия: величина эффекта отдельно дана в Cohen d. После Holm значимых шкал 1 из 15. При малой выборке, ненормальности или порядковых шкалах выводы t-тестов требуют осторожности; группы могут различаться также по классу и профилю, поэтому причинный эффект пола не установлен.

## Итоговый вывод
Статистику и p-value показываем для каждой шкалы. Наиболее и наименее выраженные различия сравниваем по абсолютному стандартизованному эффекту Cohen d, чтобы шкалы с разными единицами были сопоставимы.

In [ ]:
for label, idx in [('Сильнее всего', psy_tests.Cohen_d.abs().idxmax()),
                   ('Слабее всего', psy_tests.Cohen_d.abs().idxmin())]:
    row = psy_tests.loc[idx]
    print(f"{label}: {row['показатель']}; средние юношей/девушек "
          f"{row.mean_A:.3f}/{row.mean_B:.3f}; d={row.Cohen_d:.3f}; "
          f"t={row.t:.4f}; p={row.p:.6g}.")
print('Значимые шкалы до поправки:', ', '.join(psy_tests.loc[psy_tests.p < .05, 'показатель']))
print('Значимые шкалы после Holm:', ', '.join(psy_tests.loc[psy_tests.p_Holm < .05, 'показатель']))
